In [106]:
!pip install -U weaviate-client sentence-transformers rank-bm25 openai python-dotenv pyyaml loguru tqdm pandas numpy


[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip
c:\Users\Dunith Munasinghe\Desktop\ledger-mind\Annual-Report-Reading-RAG-Finetunning\venv\Lib\site-packages\IPython\utils\_process_win32.py:138: ResourceWarning: unclosed file <_io.BufferedWriter name=3>
  res = process_handler(cmd, _system_body)
c:\Users\Dunith Munasinghe\Desktop\ledger-mind\Annual-Report-Reading-RAG-Finetunning\venv\Lib\site-packages\IPython\utils\_process_win32.py:138: ResourceWarning: unclosed file <_io.BufferedReader name=4>
  res = process_handler(cmd, _system_body)
c:\Users\Dunith Munasinghe\Desktop\ledger-mind\Annual-Report-Reading-RAG-Finetunning\venv\Lib\site-packages\IPython\utils\_process_win32.py:138: ResourceWarning: unclosed file <_io.BufferedReader name=5>
  res = process_handler(cmd, _system_body)


In [107]:
import os
print(os.getcwd())

c:\Users\Dunith Munasinghe\Desktop\ledger-mind\Annual-Report-Reading-RAG-Finetunning\notebooks


In [108]:
pip install sentence-transformers

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [109]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

import src.rag.embedder as embedder_module
import src.rag.weaviate_indexer as indexer_module
import src.rag.librarian as librarian_module

query_librarian = librarian_module.query_librarian
BGEEmbedder = embedder_module.BGEEmbedder
WeaviateIndexer = indexer_module.WeaviateIndexer

In [110]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
sys.path.append(str(PROJECT_ROOT))


print("Project root:", PROJECT_ROOT)

Project root: c:\Users\Dunith Munasinghe\Desktop\ledger-mind\Annual-Report-Reading-RAG-Finetunning\notebooks


In [111]:
from dotenv import load_dotenv
import os

load_dotenv()

print("OpenAI key loaded:", os.getenv("OPENAI_API_KEY") is not None)
print("Weaviate URL loaded:", os.getenv("WEAVIATE_URL") is not None)
print("Weaviate API key loaded:", os.getenv("WEAVIATE_API_KEY") is not None)

OpenAI key loaded: True
Weaviate URL loaded: True
Weaviate API key loaded: True


In [112]:
Path("data/processed/chunks.json")

WindowsPath('data/processed/chunks.json')

In [113]:
import json
from pathlib import Path

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

chunks_path = PROJECT_ROOT / "data" / "processed" / "chunks.json"

assert chunks_path.exists(), f"chunks.json not found at: {chunks_path}"

with open(chunks_path, "r", encoding="utf-8") as f:
    chunks = json.load(f)

print("Project root:", PROJECT_ROOT)
print("Number of chunks:", len(chunks))
print("Keys:", chunks[0].keys())
print(chunks[0])

Project root: c:\Users\Dunith Munasinghe\Desktop\ledger-mind\Annual-Report-Reading-RAG-Finetunning
Number of chunks: 522
Keys: dict_keys(['chunk_id', 'page_start', 'page_end', 'section_title', 'text', 'char_count'])
{'chunk_id': 'uber_2024_p2_c001', 'page_start': 2, 'page_end': 2, 'section_title': 'Uber’s Mission', 'text': 'Uber’s Mission We reimagine the way the world moves for the better We are Uber. The go-getters. The kind of people who are relentless about our mission to help people go anywhere and get anything and earn their way. Movement is what we power. It’s our lifeblood. It runs through our veins. It’s what gets us out of bed each morning. It pushes us to constantly reimagine how we can move better. For you. For all the places you want to go. For all the things you want to get. For all the ways you want to earn. Across the entire world. In real time. At the incredible speed of now.', 'char_count': 579}


In [114]:
from src.rag.embedder import BGEEmbedder
from src.rag.weaviate_indexer import WeaviateIndexer

embedder = BGEEmbedder(
    model_name="BAAI/bge-small-en-v1.5",
    normalize_embeddings=True,
)

indexer = WeaviateIndexer(
    collection_name="UberAnnualReportChunks",
    embedder=embedder,
    chunks_path="data/processed/chunks.json",
)

try:
    indexer.index_chunks(reset=True)
finally:
    indexer.close()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

c:\Users\Dunith Munasinghe\Desktop\ledger-mind\Annual-Report-Reading-RAG-Finetunning\venv\Lib\site-packages\weaviate\warnings.py:196: DeprecationWarning: Dep024: You are using the `vectorizer_config` argument in `collection.config.create()`, which is deprecated.
            Use the `vector_config` argument instead.
            
  warnings.warn(


Batches:   0%|          | 0/17 [00:00<?, ?it/s]

Indexing chunks: 100%|██████████| 522/522 [00:00<00:00, 5749.88it/s]


Indexed 522 chunks into Weaviate collection: UberAnnualReportChunks


In [115]:
from src.rag.embedder import BGEEmbedder
from src.rag.retriever import LibrarianRetriever

embedder = BGEEmbedder(model_name="BAAI/bge-small-en-v1.5")

retriever = LibrarianRetriever(
    collection_name="UberAnnualReportChunks",
    embedder=embedder,
    chunks_path="data/processed/chunks.json",
)

try:
    results = retriever.retrieve(
        question="What are Uber's business segments?",
        dense_top_k=10,
        bm25_top_k=10,
        rrf_k=60,
        fused_top_k=5,
    )

    for i, item in enumerate(results, start=1):
        print("=" * 80)
        print("Rank:", i)
        print("Chunk ID:", item["chunk_id"])
        print("Page:", item["page_start"])
        print("RRF:", item["rrf_score"])
        print(item["text"][:500])
finally:
    retriever.close()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Rank: 1
Chunk ID: uber_2024_p78_c325
Page: 78
RRF: 0.01639344262295082
UBER TECHNOLOGIES, INC
Rank: 2
Chunk ID: uber_2024_p35_c153
Page: 35
RRF: 0.01639344262295082
. In addition, in connection with any planned or future acquisitions, we may acquire businesses that have differing licenses and other arrangements that may be challenged by tax authorities for not being at arm’s-length or that are otherwise potentially less tax efficient than our licenses and arrangements. Any subsequent integration or continued operation of such acquired businesses may result in an increased effective tax rate in certain jurisdictions or potential indirect tax costs, which cou
Rank: 3
Chunk ID: uber_2024_p8_c014
Page: 8
RRF: 0.016129032258064516
. Uber is also developing technologies designed to provide new solutions to solve everyday problems. Our technology is available in over 70 countries around the world, principally in the United States (“U.S.”) and Canada, Latin America, Europe (excluding Russia), 

In [116]:
from src.rag.reranker import CrossEncoderReranker

reranker = CrossEncoderReranker(
    model_name="cross-encoder/ms-marco-MiniLM-L-6-v2"
)

reranked = reranker.rerank(
    question="What are Uber's business segments?",
    candidates=results,
    top_k=3,
)

for i, item in enumerate(reranked, start=1):
    print("=" * 80)
    print("Reranked:", i)
    print("Chunk ID:", item["chunk_id"])
    print("Page:", item["page_start"])
    print("RRF:", item["rrf_score"])
    print("Rerank Score:", item["rerank_score"])
    print(item["text"][:500])

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Reranked: 1
Chunk ID: uber_2024_p8_c014
Page: 8
RRF: 0.016129032258064516
Rerank Score: 5.191236972808838
. Uber is also developing technologies designed to provide new solutions to solve everyday problems. Our technology is available in over 70 countries around the world, principally in the United States (“U.S.”) and Canada, Latin America, Europe (excluding Russia), the Middle East, Africa, and Asia Pacific (“APAC”, excluding China and Southeast Asia). Our Segments As of December 31, 2024, we had three operating and reportable segments: Mobility, Delivery and Freight. Mobility, Delivery and Freight
Reranked: 2
Chunk ID: uber_2024_p8_c013
Page: 8
RRF: 0.015873015873015872
Rerank Score: 2.9111499786376953
PART I ITEM 1. BUSINESS Overview Uber Technologies, Inc. (“Uber,” the “Company,” “we,” “our,” or “us”) is a technology platform that uses a massive network, leading technology, operational excellence and product expertise to power movement from point A to point B. We develop and operat

In [117]:
from pathlib import Path
PROJECT_ROOT = Path("c:/Users/Dunith Munasinghe/Desktop/ledger-mind/Annual-Report-Reading-RAG-Finetunning")
print((PROJECT_ROOT / "configs/rag_config.yaml").exists())

True


In [120]:
import importlib
import src.rag.librarian as librarian_module

importlib.reload(librarian_module)

result = librarian_module.query_librarian("What are Uber's business segments?")

result

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

{'question': "What are Uber's business segments?",
 'answer': "Uber's business segments, as of December 31, 2024, are Mobility, Delivery, and Freight (Source: uber_2024_p8_c014 | Page: 8).",
 'sources': [{'page': 8,
   'page_end': 8,
   'chunk_id': 'uber_2024_p8_c014',
   'rrf_score': 0.016129032258064516,
   'rerank_score': 5.19123649597168},
  {'page': 8,
   'page_end': 8,
   'chunk_id': 'uber_2024_p8_c013',
   'rrf_score': 0.015873015873015872,
   'rerank_score': 2.9111502170562744},
  {'page': 86,
   'page_end': 86,
   'chunk_id': 'uber_2024_p86_c338',
   'rrf_score': 0.015384615384615385,
   'rerank_score': 2.582770824432373},
  {'page': 78,
   'page_end': 78,
   'chunk_id': 'uber_2024_p78_c325',
   'rrf_score': 0.01639344262295082,
   'rerank_score': -1.488434910774231},
  {'page': 53,
   'page_end': 53,
   'chunk_id': 'uber_2024_p53_c240',
   'rrf_score': 0.015151515151515152,
   'rerank_score': -2.0078816413879395}]}

In [121]:
questions = [
    "What are Uber's business segments?",
    "What does Uber say about Mobility?",
    "What does Uber say about Delivery?",
    "What are the main risks mentioned in the report?",
    "How does Uber generate revenue?",
]

for q in questions:
    print("=" * 100)
    print("QUESTION:", q)

    result = query_librarian(q)

    print("ANSWER:")
    print(result["answer"])

    print("SOURCES:")
    for source in result["sources"]:
        print(source)

QUESTION: What are Uber's business segments?


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

ANSWER:
Uber's business segments, as of December 31, 2024, are Mobility, Delivery, and Freight (source: page 8).
SOURCES:
{'page': 8, 'page_end': 8, 'chunk_id': 'uber_2024_p8_c014', 'rrf_score': 0.016129032258064516, 'rerank_score': 5.19123649597168}
{'page': 8, 'page_end': 8, 'chunk_id': 'uber_2024_p8_c013', 'rrf_score': 0.015873015873015872, 'rerank_score': 2.9111502170562744}
{'page': 86, 'page_end': 86, 'chunk_id': 'uber_2024_p86_c338', 'rrf_score': 0.015384615384615385, 'rerank_score': 2.582770824432373}
{'page': 78, 'page_end': 78, 'chunk_id': 'uber_2024_p78_c325', 'rrf_score': 0.01639344262295082, 'rerank_score': -1.488434910774231}
{'page': 53, 'page_end': 53, 'chunk_id': 'uber_2024_p53_c240', 'rrf_score': 0.015151515151515152, 'rerank_score': -2.0078816413879395}
QUESTION: What does Uber say about Mobility?


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

ANSWER:
The main risks mentioned in the report include:

1. Classification of Drivers as employees instead of independent contractors (Page 13).
2. Market risks such as interest rate risk, investment risk, and foreign currency risk (Page 72).
3. Cybersecurity threats, including breaches and cyberattacks (Page 50).
4. Risks related to data collection, use, and privacy (Page 50).
5. Climate change risks, including physical and transitional risks (Page 30).
SOURCES:
{'page': 13, 'page_end': 13, 'chunk_id': 'uber_2024_p13_c036', 'rrf_score': 0.02946236559139785, 'rerank_score': 2.971851348876953}
{'page': 72, 'page_end': 72, 'chunk_id': 'uber_2024_p72_c308', 'rrf_score': 0.028381642512077296, 'rerank_score': 1.102717638015747}
{'page': 50, 'page_end': 50, 'chunk_id': 'uber_2024_p50_c230', 'rrf_score': 0.01639344262295082, 'rerank_score': -0.027249686419963837}
{'page': 30, 'page_end': 30, 'chunk_id': 'uber_2024_p30_c124', 'rrf_score': 0.014705882352941176, 'rerank_score': -0.83979517221450

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

ANSWER:
Uber generates revenue primarily through its Mobility and Delivery services, as well as Freight services. Revenue is derived from service fees charged to Drivers and Merchants for each transaction facilitated through the platform, as defined in Master Services Agreements (MSA) (page 91). Additionally, Uber offers subscription memberships to end-users, which contribute to Mobility and Delivery revenue (page 98). In the last reported year, Uber's revenue was $44.0 billion, driven by an increase in Gross Bookings from Mobility and Delivery Trip volumes (page 54).
SOURCES:
{'page': 91, 'page_end': 91, 'chunk_id': 'uber_2024_p91_c362', 'rrf_score': 0.015625, 'rerank_score': 2.2898311614990234}
{'page': 54, 'page_end': 54, 'chunk_id': 'uber_2024_p54_c245', 'rrf_score': 0.015873015873015872, 'rerank_score': 2.1824193000793457}
{'page': 79, 'page_end': 79, 'chunk_id': 'uber_2024_p79_c329', 'rrf_score': 0.015384615384615385, 'rerank_score': 2.0190682411193848}
{'page': 98, 'page_end': 9